In [1]:
import re
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
#Load your dataset
DATA_PATH = r"C:\TARUN\DATASET\NLP\Chatbot\dialogs.txt"

df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    header=None,
    names=["question", "answer"],
    on_bad_lines="skip"
)

df = df.dropna(subset=["question", "answer"]).copy()

df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
].reset_index(drop=True)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head(10))


Dataset loaded successfully!
Dataset shape: (3725, 2)


,question,answer
0,"hi, how are you doing?",i'm fine. how about yourself?
1,i'm fine. how about yourself?,i'm pretty good. thanks for asking.
2,i'm pretty good. thanks for asking.,no problem. so how have you been?
3,no problem. so how have you been?,i've been great. what about you?
4,i've been great. what about you?,i've been good. i'm in school right now.
5,i've been good. i'm in school right now.,what school do you go to?
6,what school do you go to?,i go to pcc.
7,i go to pcc.,do you like it there?
8,do you like it there?,it's okay. it's a really big campus.
9,it's okay. it's a really big campus.,good luck with school.


In [3]:
#Clean questions and train TF-IDF
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

df["question_clean"] = df["question"].apply(clean_text)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    strip_accents="unicode",
    sublinear_tf=True
)

question_vectors = vectorizer.fit_transform(
    df["question_clean"]
)

print("TF-IDF model trained successfully!")
print("Total questions:", len(df))
print("Vector shape:", question_vectors.shape)


TF-IDF model trained successfully!
Total questions: 3725
Vector shape: (3725, 12589)


In [4]:
#Create the chatbot function
def chatbot_reply(message, threshold=0.25):
    if not message or not message.strip():
        return "Please type a message."

    message_clean = clean_text(message)
    message_vector = vectorizer.transform([message_clean])

    similarities = cosine_similarity(
        message_vector,
        question_vectors
    ).flatten()

    if similarities.size == 0 or similarities.max() == 0:
        return "Sorry, I couldn't find a suitable answer."

    best_index = similarities.argmax()
    best_score = similarities[best_index]

    print(f"Similarity score: {best_score:.3f}")

    if best_score < threshold:
        return (
            "Sorry, I couldn't find a suitable answer. "
            "Please try another question."
        )

    return df.iloc[best_index]["answer"]

In [5]:
#Test the chatbot
test_questions = [
    "Hi",
    "How are you?",
    "What is your name?",
    "Where are you from?",
    "Thank you",
    "What is quantum physics?",
    "How is the weather on Mars?"
]

for question in test_questions:
    print("You:", question)
    print("Bot:", chatbot_reply(question))
    print("-" * 50)

You: Hi
Similarity score: 0.443
Bot: i'm fine. how about yourself?
--------------------------------------------------
You: How are you?
Similarity score: 0.639
Bot: i started shopping at the dollar store.
--------------------------------------------------
You: What is your name?
Similarity score: 0.720
Bot: english.
--------------------------------------------------
You: Where are you from?
Similarity score: 0.750
Bot: i have to walk the dog.
--------------------------------------------------
You: Thank you
Similarity score: 1.000
Bot: i'm so happy for you.
--------------------------------------------------
You: What is quantum physics?
Similarity score: 0.602
Bot: it's the force that pulls everything down.
--------------------------------------------------
You: How is the weather on Mars?
Similarity score: 0.481
Bot: the forecast says that it will be warm on the weekend.
--------------------------------------------------


In [6]:
#Start chatting with the bot
print("===================================")
print("          MY NLP CHATBOT")
print("===================================")
print("Type 'exit' to stop chatting.\n")

while True:
    user_message = input("You: ").strip()

    if user_message.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye! Have a nice day.")
        break

    if not user_message:
        print("Bot: Please type a message.\n")
        continue

    print("Bot:", chatbot_reply(user_message))
    print()


          MY NLP CHATBOT
Type 'exit' to stop chatting.



You:  hi


Similarity score: 0.443
Bot: i'm fine. how about yourself?



You:  how are you


Similarity score: 0.639
Bot: i started shopping at the dollar store.



You:  exit


Bot: Goodbye! Have a nice day.


In [7]:
#Save your trained chatbot model
model_data = {
    "vectorizer": vectorizer,
    "question_vectors": question_vectors,
    "questions": df["question_clean"].tolist(),
    "answers": df["answer"].tolist(),
    "threshold": 0.25
}

joblib.dump(model_data, "chatbot_model.pkl")

print("Chatbot model saved successfully!")


Chatbot model saved successfully!


In [8]:
#Load the saved model later
model_data = joblib.load("chatbot_model.pkl")
vectorizer = model_data["vectorizer"]
question_vectors = model_data["question_vectors"]
threshold = model_data.get("threshold", 0.25)

df = pd.DataFrame({
    "question_clean": model_data["questions"],
    "answer": model_data["answers"]
})

print("Saved chatbot model loaded successfully!")


Saved chatbot model loaded successfully!
